In [ ]:
# Base Model vs. Fine-Tuned Model: A/B Evaluation

**Objective:**
This notebook conducts a direct comparative analysis between the raw, pre-trained `Gemma-4-26b` model and our newly fine-tuned categorization model. 

**Methodology:**
To evaluate both models efficiently without exceeding AWS GPU VRAM limits, we load the base model and attach our trained LoRA adapters. Using dynamic adapter toggling, we pass the exact same product strings to the "Raw" network and the "Trained" network. We then measure their respective outputs against the verified ground-truth data from the Mercari `Categorized.xlsx` dataset.

In [ ]:
# Install required dependencies
!pip install -q transformers peft torch pandas openpyxl huggingface_hub

import os
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# --- Configuration & File Paths ---
INPUT_DIR = 'path/to/your/output_directory/Split_Files'
EXCEL_PATH = os.path.join(INPUT_DIR, 'Categorized.xlsx')
TAXONOMY_PATH = os.path.join(INPUT_DIR, 'master_taxonomy.json')

# Model Paths
BASE_MODEL_ID = "google/gemma-4-26b"
ADAPTER_PATH = "./aws_weights/gemma4_26b_100_percent" # Your previously trained weights

# Security & API Setup
HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN_HERE"

# Dataset Columns
TITLE_COLUMN = "item_name"
TRUE_C1_COL = "category_1"
TRUE_C2_COL = "category_2"
TRUE_C3_COL = "category_3"

print("Loading taxonomy and reference dataset...")
try:
    with open(TAXONOMY_PATH, "r", encoding="utf-8") as f:
        taxonomy = json.load(f)
    df = pd.read_excel(EXCEL_PATH)
    print(f"✅ Loaded taxonomy tree and {len(df)} reference products.")
except Exception as e:
    print(f"❌ Initialization Error: {e}")

In [ ]:
# --- Initialize Gemma 4 & Attach Adapters ---
print(f"Initializing {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN)

# Load base model in bfloat16 format
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID, 
    torch_dtype=torch.bfloat16, 
    device_map="auto", 
    token=HF_TOKEN
)

print(f"Injecting trained adapters from {ADAPTER_PATH}...")
# Wrap base model with PEFT to allow dynamic adapter toggling
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print("✅ Dual-State Model ready for A/B Testing.")

# --- The Inference Function ---
def ask_gemma_category(product: str, options: list, level_name: str, parent_context: str = "", use_raw_model: bool = False) -> str:
    """Queries the model. Toggles to raw weights if use_raw_model=True."""
    options_list_str = "\n".join([f"- {opt}" for opt in options])
    context_str = f"Parent Category Path: {parent_context}\n" if parent_context else ""
    
    system_instruction = (
        "You are a strict data classification pipeline.\n"
        "Output ONLY the exact category name from the provided 'Valid Categories' list.\n"
        "NO explanations. NO conversational text."
    )
    
    user_prompt = (
        f"{context_str}Valid {level_name} Categories:\n{options_list_str}\n\n"
        f"Task: Classify this product into exactly one of the categories above.\n"
        f"Product: \"{product}\"\nCategory:"
    )

    messages = [{"role": "user", "content": f"{system_instruction}\n\n{user_prompt}"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Dynamic Context Manager: Disables adapter to test base model performance
    context_manager = model.disable_adapter() if use_raw_model else torch.no_grad()
    
    with context_manager:
        outputs = model.generate(
            **inputs, 
            max_new_tokens=20, 
            temperature=0.01, 
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
        
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    prediction = response.replace('"', '').replace("'", "").lstrip('- ').strip()
    return prediction

# --- The Smart Matcher & Classifier ---
def classify_product(product: str, taxonomy_tree: dict, use_raw: bool = False) -> tuple:
    """Executes hierarchical classification with fuzzy matching fallback."""
    cat1, cat2, cat3 = "Other", "Other", "Other"
    
    def find_best_match(pred, valid_options):
        if not pred: return None
        pred_clean = pred.strip().lower()
        for opt in valid_options:
            if str(opt).strip().lower() == pred_clean: return opt
        for opt in valid_options:
            if str(opt).strip().lower() in pred_clean: return opt
        return None

    # L1
    l1_options = list(taxonomy_tree.keys())
    l1_pred = ask_gemma_category(product, l1_options, "Level 1", use_raw_model=use_raw)
    cat1_match = find_best_match(l1_pred, l1_options)
    
    if cat1_match:
        cat1 = cat1_match
        # L2
        l2_data = taxonomy_tree[cat1]
        l2_options = list(l2_data.keys()) if isinstance(l2_data, dict) else l2_data
        l2_pred = ask_gemma_category(product, l2_options, "Level 2", cat1, use_raw_model=use_raw)
        cat2_match = find_best_match(l2_pred, l2_options)
        
        if cat2_match and isinstance(l2_data, dict):
            cat2 = cat2_match
            # L3
            l3_data = l2_data[cat2]
            if isinstance(l3_data, (dict, list)):
                l3_options = list(l3_data.keys()) if isinstance(l3_data, dict) else l3_data
                if len(l3_options) > 0:
                    parent_path = f"{cat1} > {cat2}"
                    l3_pred = ask_gemma_category(product, l3_options, "Level 3", parent_path, use_raw_model=use_raw)
                    cat3_match = find_best_match(l3_pred, l3_options)
                    if cat3_match:
                        cat3 = cat3_match

    return cat1, cat2, cat3

In [ ]:
# --- Single Product A/B Test ---

print("=== Single Item Diagnostic A/B Validation ===")
random_row = df.sample(1).iloc[0]

test_product = str(random_row[TITLE_COLUMN])
true_path = [str(random_row[TRUE_C1_COL]), str(random_row[TRUE_C2_COL]), str(random_row[TRUE_C3_COL])]
actual_path_str = ' > '.join(true_path)

print(f"📦 Product: '{test_product}'\n")

# 1. Test Raw Model
print("🤖 Testing Raw Gemma 4 (Base)...")
raw_c1, raw_c2, raw_c3 = classify_product(test_product, taxonomy, use_raw=True)
raw_path_str = f"{raw_c1} > {raw_c2} > {raw_c3}"

# 2. Test Trained Model
print("🤖 Testing Trained Gemma 4 (Adapters Enabled)...")
trained_c1, trained_c2, trained_c3 = classify_product(test_product, taxonomy, use_raw=False)
trained_path_str = f"{trained_c1} > {trained_c2} > {trained_c3}"

# Output Comparison
print("\n" + "="*70)
print("📊 PERFORMANCE RESULTS:")
print(f"   Actual DB Label   : {actual_path_str}")
print("-" * 70)
print(f"   Raw Model Pred    : {raw_path_str}")
print(f"   Trained Model Pred: {trained_path_str}")
print("="*70)

In [ ]:
# --- 10 Item Batch A/B Benchmarking ---

print("="*80)
print("🚀 STARTING BATCH EVALUATION (10 ITEMS) - RAW VS TRAINED")
print("="*80)

sample_size = 10
random_rows = df.sample(sample_size, random_state=42) # Fixed seed for reproducible benchmarks

# Accuracy Trackers
raw_correct = {'L1': 0, 'L2': 0, 'L3': 0}
trained_correct = {'L1': 0, 'L2': 0, 'L3': 0}

for index, row in random_rows.iterrows():
    test_product = str(row[TITLE_COLUMN])
    true_c1, true_c2, true_c3 = str(row[TRUE_C1_COL]), str(row[TRUE_C2_COL]), str(row[TRUE_C3_COL])
    
    print(f"\n📦 PRODUCT: '{test_product[:50]}...'")
    
    # Inference
    r_c1, r_c2, r_c3 = classify_product(test_product, taxonomy, use_raw=True)
    t_c1, t_c2, t_c3 = classify_product(test_product, taxonomy, use_raw=False)
    
    # Scoring Raw
    if str(r_c1) == true_c1: raw_correct['L1'] += 1
    if str(r_c2) == true_c2: raw_correct['L2'] += 1
    if str(r_c3) == true_c3: raw_correct['L3'] += 1
        
    # Scoring Trained
    if str(t_c1) == true_c1: trained_correct['L1'] += 1
    if str(t_c2) == true_c2: trained_correct['L2'] += 1
    if str(t_c3) == true_c3: trained_correct['L3'] += 1

    print(f"   Actual  : {true_c1} > {true_c2} > {true_c3}")
    print(f"   Raw     : {r_c1} > {r_c2} > {r_c3}")
    print(f"   Trained : {t_c1} > {t_c2} > {t_c3}")
    print("-" * 80)

# Final Output Report
print("\n" + "="*80)
print("📊 FINAL A/B PERFORMANCE METRICS (Out of 10)")
print("="*80)
print(f"   [RAW MODEL ACCURACY]")
print(f"   Level 1: {raw_correct['L1']}/10 ({(raw_correct['L1']/10)*100:.0f}%)")
print(f"   Level 2: {raw_correct['L2']}/10 ({(raw_correct['L2']/10)*100:.0f}%)")
print(f"   Level 3: {raw_correct['L3']}/10 ({(raw_correct['L3']/10)*100:.0f}%)\n")

print(f"   [TRAINED MODEL ACCURACY]")
print(f"   Level 1: {trained_correct['L1']}/10 ({(trained_correct['L1']/10)*100:.0f}%)")
print(f"   Level 2: {trained_correct['L2']}/10 ({(trained_correct['L2']/10)*100:.0f}%)")
print(f"   Level 3: {trained_correct['L3']}/10 ({(trained_correct['L3']/10)*100:.0f}%)")
print("="*80)